# RNA-seq Standalone Pipeline (v2)
### Tidy → annotate → facility batch → whole blood → pre-treatment → one sample per patient → RPKM / TPM

**What this notebook does.** It takes the compiled RNA-seq count workbooks (samples as columns, genes as rows, sample metadata mixed in) and produces an analysis-ready cohort with raw counts, a **batch column**, and demographics.

<div class="alert alert-warning">
<b>What changed from v1, and why.</b> These changes come from reading the NIH-NLM endotypes-transcriptomics notebooks (40, 41, 46-49b):
<ol>
<li><b>Every sheet of every workbook is read.</b> v1 read only the first sheet, which for the Goldbach-Mansky workbook is NIAMS, so the other three sequencing facilities (BCM-HGSC, NCI-FNL, NIAID-NCI) were never loaded.</li>
<li><b>A <code>batch</code> column is built</b> from <code>Sequencing_Facility</code> (NIAMS is split by <code>Sequencing_Cohort</code>), and a batch × diagnosis table shows which batches can be separated from diagnosis at all.</li>
<li><b>Raw counts are the primary output.</b> RPKM / TPM are display values. Differential expression should use raw counts with batch as a model term (notebook 4).</li>
<li><b>Two cohorts are written:</b> <i>all eligible samples</i> (for differential expression with a patient block) and <i>one per patient</i> (for WGCNA, which needs independent samples).</li>
<li>Paths now use <code>/mnt/file-systems</code> (with a hyphen) and results go to a <b>new folder</b>, <code>rnaseq_out_v2</code>, so old and new results cannot be mixed up.</li>
</ol>
</div>

**Roadmap**

| Part | Step | Question it answers |
|---|---|---|
| 1 | Configuration | where is everything, what are the rules? |
| 2 | Helpers | how do we compare IDs and read every sheet of a big Excel file fast? |
| 3 | Patient metadata | who are the patients? |
| 4 | Load RNA-seq | which samples, sheets and genes do we have? |
| 5 | Whole blood filter | is it really whole blood? |
| 6 | Dates | when was it drawn? |
| 7 | Join metadata | whose sample is it? |
| 8 | Treatment status | pre- or post-treatment? |
| 9 | Eligible set, one per patient, batch table | who is in, and can batch be separated from diagnosis? |
| 10 | RPKM / TPM | length-normalised display values |
| 11 | Write outputs | CSVs and a final audit |

<div class="alert alert-info">
<b>Design principle: filter by flags, never by deletion.</b> Every sample keeps flag columns, and <code>rnaseq_samples_ALL_with_flags.csv</code> preserves everything, so you can always see why a sample was excluded.
</div>

<div class="alert alert-warning">
<b>Privacy.</b> Birth date is read only to compute age, then dropped. Outputs contain no DOB and no raw dates. All outputs are CSV.
</div>

---
# Part 1 · Configuration

🎯 **Goal:** every path, rule and cohort decision lives in this cell.

💡 **Why:** the scientific *decisions* are named variables, not buried logic. `OUT_DIR` is a **new** folder (`rnaseq_out_v2`); `LEGACY_LENGTHS` lets the notebook reuse the gene-length table from the old run instead of downloading a GTF again.

In [ ]:
# === RNA-seq standalone v2: all sheets, facility batch, whole blood, pre-treatment, one per patient
import os, re, glob, gzip, shutil, datetime, urllib.request
import numpy as np, pandas as pd

MNT = "/mnt/file-systems/"
RNA_FILES = [
    MNT + "HC_blood_all_samples.xlsx",
    MNT + "Goldbach-Mansky_compiled_RNA-Seq_GeneCounts_CANDLE_AND_SAVI_SUBSET_2026_09_22.xlsx",
]
METADATA_PATH = MNT + "MetadataPatients_NoPHI.xlsx"
METADATA_EXTRA = []        # e.g. a healthy-volunteer sheet with the same headers
RESULTS   = "/home/jovyan/session_data/endotype_results"       # the ONE results folder (outside the read-only S3 mount)
OUT_DIR   = RESULTS + "/rnaseq_out_v2"
CACHE_DIR = "/home/jovyan/rnaseq_cache"       # xlsx -> csv cache, kept off the mount
LEGACY_LENGTHS = MNT + "rnaseq_out/gene_lengths.csv"   # reused if present, so no new GTF download

KEEP_ONE_PER_PATIENT = "earliest"            # or "deepest" (most total counts)
REQUIRE_PRE_FOR_DX = r"CANDLE|SAVI"          # these cohorts have PRE/POST labels: insist on PRE
BLOOD_VALUES = {"blood", "whole blood", "wholeblood", "wb"}   # exact match; PBMC never passes
GTF_PATH = ""                                # blank = auto-find, then try download
GTF_URL = ("https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/"
           "release_19/gencode.v19.annotation.gtf.gz")
DX_MAP = {"hc whole blood": "HC",
          "relative-hc": "HC",
          "undifferentiated interferonopathy": "Undifferentiated Interferonopathy"}
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
assert os.access(OUT_DIR, os.W_OK), f"{OUT_DIR} is not writable"

---
# Part 2 · Helpers

🎯 **Goal:** an ID normaliser and a cached reader that returns **every sheet** of a workbook.

💡 **Why:**
- **`normalize_id`** makes `O-6033`, `o6033` and `O6033` compare equal, strips Excel's `.0`, and maps RNA's `CHLD0047` to the metadata's `LD0047`.
- **`read_workbook_cached`** converts each sheet to its own CSV once (cache folder, off the mount) and re-converts only when the Excel file is newer. Everything is read as text so `0047` keeps its zeros. **This is the fix for the first-sheet-only problem.**
- `DATE_MIN/DATE_MAX` is a sanity window: any parsed date outside 2008-today is treated as a typo.

In [ ]:
# ---- helpers ---------------------------------------------------------------
def normalize_id(x):
    s = "" if x is None else str(x).strip()
    if s.lower() in ("nan", "none", "nat", ""): return ""
    s = re.sub(r"\.0$", "", s)
    s = s.upper().replace("-", "").replace(" ", "").replace("_", "")
    if s.startswith("CHLD"):                 # CHLD0047 (RNA files) == LD0047 (metadata)
        s = s[2:]
    return s

def norm_colname(c): return re.sub(r"[^a-z0-9]", "", str(c).lower())

def _safe(s): return re.sub(r"[^A-Za-z0-9]+", "_", str(s)).strip("_")

DATE_MIN, DATE_MAX = pd.Timestamp("2008-01-01"), pd.Timestamp.today().normalize()

def read_workbook_cached(path):
    """Return {sheet name: DataFrame of text} for EVERY sheet of an .xlsx (or the one csv).
    Each sheet is converted to csv once, in CACHE_DIR, and re-converted only if the Excel file is newer."""
    stem = os.path.splitext(os.path.basename(path))[0]
    if path.lower().endswith(".csv"):
        return {stem: pd.read_csv(path, dtype=object, keep_default_na=False, low_memory=False)}
    xl = pd.ExcelFile(path)
    out = {}
    for sh in xl.sheet_names:
        csv_path = os.path.join(CACHE_DIR, f"{stem}__{_safe(sh)}.csv")
        if (not os.path.exists(csv_path)) or os.path.getmtime(csv_path) < os.path.getmtime(path):
            print(f"converting {os.path.basename(path)} [{sh}] -> csv (one time) ...")
            xl.parse(sh, dtype=object).to_csv(csv_path, index=False)
        out[sh] = pd.read_csv(csv_path, dtype=object, keep_default_na=False, low_memory=False)
    return out

---
# Part 3 · Patient metadata

🎯 **Goal:** load the master patient sheet and build fast lookups from it.

💡 **Why:** the sheet with the most columns like *enroll / diagnos / initial* is chosen automatically; `METADATA_EXTRA` appends more rows (main sheet first, so it wins on duplicate IDs); column names are matched loosely; initials map to an ID **only where they are unique**, because a wrong patient is worse than a blank one.

In [ ]:
# ---- patient metadata ------------------------------------------------------
sheets = pd.read_excel(METADATA_PATH, sheet_name=None, dtype=str)
md = max(sheets.values(), key=lambda d: sum(bool(re.search(r"enroll|diagnos|initial", str(c), re.I))
                                            for c in d.columns))
md.columns = [str(c).strip() for c in md.columns]
md = md.dropna(how="all").reset_index(drop=True)

for p in METADATA_EXTRA:                     # main sheet first, so it wins on duplicate IDs
    ex = (pd.read_excel(p, dtype=str) if p.lower().endswith((".xlsx", ".xls"))
          else pd.read_csv(p, dtype=str))
    ex.columns = [str(c).strip() for c in ex.columns]
    nm = {norm_colname(c): c for c in md.columns}
    ex = ex.rename(columns={c: nm[norm_colname(c)] for c in ex.columns if norm_colname(c) in nm})
    md = pd.concat([md, ex.dropna(how="all").reindex(columns=md.columns)], ignore_index=True)
    print(f"extra metadata: {p} (+{len(ex)} rows)")

def fc(*names):
    nm = {norm_colname(c): c for c in md.columns}
    return next((nm[norm_colname(n)] for n in names if norm_colname(n) in nm), None)

COL = {"enroll": fc("Enrollment ID"), "alt": fc("Alternate Enrollment ID"),
       "initials": fc("Initials"), "dob": fc("Birth Date", "DOB"), "sex": fc("Sex", "Gender"),
       "race": fc("Race"), "ethnicity": fc("Ethnicity"), "status": fc("Study Status"),
       "dx1": fc("Diagnosis1"), "dx2": fc("Diagnosis2"), "note": fc("Note", "Notes")}
print("metadata:", len(md), "rows; columns not found:", [k for k, v in COL.items() if v is None])

meta_lookup, by_ini = {}, {}
for i, r in md.iterrows():
    for kc in (COL["enroll"], COL["alt"]):
        k = normalize_id(r.get(kc)) if kc else ""
        if k: meta_lookup.setdefault(k, i)
    ini = str(r.get(COL["initials"]) or "").strip().upper() if COL["initials"] else ""
    eid = normalize_id(r.get(COL["enroll"])) if COL["enroll"] else ""
    if ini and eid: by_ini.setdefault(ini, set()).add(eid)
AUTO_INITIALS_MAP = {k: next(iter(v)) for k, v in by_ini.items() if len(v) == 1}

def meta_value(i, key):
    c = COL.get(key)
    if i < 0 or not c: return ""
    v = md.at[i, c]
    return "" if pd.isna(v) else str(v).strip()

---
# Part 4 · Load every sheet of every workbook

🎯 **Goal:** one **sample table** (one row per sample) and one **genes × samples count matrix**, built from *every sheet that has the expected layout*.

💡 **Why this reshaping is needed:** in the compiled files, rows labelled `Raw_Count` are gene counts, but **all other rows are sample metadata** (diagnosis, treatment, facility...) stored as if they were genes. The loader splits them and transposes the metadata so each sample becomes a row.

**Each sheet is one "unit"** (for the Goldbach-Mansky workbook, one unit per sequencing facility). Units are combined on `Gene_ID` with an **inner join**. Two checks are printed: whether every unit lists the *same genes in the same order* (the check their notebook 40 makes), and how many genes survive the join. A sheet without `Gene_ID / Gene_Name / Stat` columns is skipped and reported, not silently dropped. Repeated sample names across units raise an error.

**Compare the per-unit sample counts with their notebook 40** (BCM-HGSC 11, NCI-FNL 20, NIAID-NCI 15, NIAMS 126). Small differences can come from samples with blank counts, which are flagged below, not deleted.

In [ ]:
# ---- 1. load: split metadata rows from count rows, samples become rows ------
REQUIRED = {"Gene_ID", "Gene_Name", "Stat"}

def load_unit(df, source_file, sheet):
    raw = df.copy()
    raw.columns = [str(c).strip() for c in raw.columns]
    if not REQUIRED <= set(raw.columns):
        return None
    samples = [c for c in raw.columns if c not in REQUIRED]
    is_count = raw["Stat"].astype(str).str.strip().eq("Raw_Count")
    meta = raw.loc[~is_count, ["Stat"] + samples].set_index("Stat").T
    meta.columns = [str(c).strip() for c in meta.columns]
    meta.index.name = "sample_id"
    meta = meta.reset_index()
    meta["source_file"], meta["source_sheet"] = source_file, sheet
    cnt = raw.loc[is_count, ["Gene_ID", "Gene_Name"] + samples].copy()
    cnt[samples] = cnt[samples].apply(pd.to_numeric, errors="coerce")
    for s in samples:
        if cnt[s].isna().all():
            print(f"WARNING [{sheet}]: no numeric counts in sample {s}")
    return meta, cnt

units, metas, cnts = [], [], []
for p in RNA_FILES:
    for sheet, df in read_workbook_cached(p).items():
        res = load_unit(df, os.path.basename(p), sheet)
        if res is None:
            print(f"skipped {os.path.basename(p)} [{sheet}]: no Gene_ID / Gene_Name / Stat columns")
            continue
        m, c = res
        units.append({"file": os.path.basename(p)[:40], "sheet": sheet,
                      "samples": c.shape[1] - 2, "genes": len(c)})
        metas.append(m); cnts.append(c)
units_df = pd.DataFrame(units)
print(units_df.to_string(index=False))

ids0 = cnts[0]["Gene_ID"].tolist()
same_order = [c["Gene_ID"].tolist() == ids0 for c in cnts]
print(f"\nunits listing the same genes in the same order as the first unit: {sum(same_order)} of {len(cnts)}")

### 4.2 Combine into one count matrix and one sample table

💡 **Why standardise column names:** files label the same field differently (`Enrollment ID` vs `EnrollmentID`, `Gender` vs `Sex`). The rename map gives every field one canonical name; missing fields are created blank so later code never hits a `KeyError`. Placeholder text becomes empty. `total_counts` (library size) is computed now because it is used for tie-breaking and RPKM, and samples with **no counts at all** are flagged `has_counts = False` and excluded later with the reason recorded.

**Batch.** `batch` = the sequencing facility; for NIAMS it is the facility plus the sequencing cohort (`Batch1`, `Batch2`, or `unlabelled`). If a facility is blank, the sheet name is used instead. This is the same six-level batch their notebook 40 uses, and it is what goes into the model as a term.

In [ ]:
gene_names = cnts[0].drop_duplicates("Gene_ID").set_index("Gene_ID")["Gene_Name"]
mats = [c.drop(columns="Gene_Name").groupby("Gene_ID").sum(min_count=1) for c in cnts]
counts = pd.concat(mats, axis=1, join="inner")
print(f"genes in every unit: {len(counts)}  (per unit: {[len(m) for m in mats]})")
dup = counts.columns[counts.columns.duplicated()]
if len(dup): raise ValueError(f"sample names repeated across units: {list(dup)[:5]}")

rn = {"enrollmentid": "enrollment_id_rna", "metadatainitials": "initials_rna",
      "samplesource": "sample_source", "diagnosis": "diagnosis_rna", "treatment": "treatment",
      "sequencingfacility": "sequencing_facility", "sequencingcohort": "sequencing_cohort",
      "gender": "gender_rna", "rnasamplecollectiondate": "rna_collection_date"}
rmeta = pd.concat(metas, ignore_index=True)
rmeta = rmeta.rename(columns=lambda c: rn.get(norm_colname(c), c))
for c in rn.values():
    if c not in rmeta.columns: rmeta[c] = ""
rmeta = rmeta.fillna("").replace({"-": "", "NaN": "", "nan": ""})
rmeta["total_counts"] = rmeta["sample_id"].map(counts.sum(min_count=1))
rmeta["has_counts"] = rmeta["total_counts"].fillna(0) > 0
print(f"\nsamples with no counts at all (excluded below): {int((~rmeta['has_counts']).sum())}",
      rmeta.loc[~rmeta["has_counts"], "sample_id"].tolist())
rmeta["dx_clean"] = rmeta["diagnosis_rna"].map(
    lambda s: DX_MAP.get(str(s).strip().lower(), str(s).strip()))

# ---- batch = facility (NIAMS split by sequencing cohort) --------------------
_blank = ["", "NA", "nan", "-"]
fac = rmeta["sequencing_facility"].astype(str).str.strip()
fac = fac.where(~fac.isin(_blank), rmeta["source_sheet"])
coh = rmeta["sequencing_cohort"].astype(str).str.strip()
coh = coh.where(~coh.isin(_blank), "unlabelled")
rmeta["facility"] = fac
rmeta["batch"] = np.where(fac.str.upper().eq("NIAMS"), "NIAMS-" + coh, fac)
print("\nsamples per batch (all samples read):")
print(rmeta.groupby(["source_sheet", "batch"]).size().rename("n").reset_index().to_string(index=False))

### 👀 Checkpoint 4: what did we load?

Left: samples per diagnosis. Right: sequencing depth per sample, log scale. Very shallow samples at the left edge are worth knowing about.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
rmeta["dx_clean"].replace("", "(blank)").value_counts().head(12).plot.barh(ax=ax[0], color="#4C78A8")
ax[0].set_title(f"Samples per diagnosis (n={len(rmeta)})"); ax[0].invert_yaxis()
ax[1].hist(np.log10(rmeta["total_counts"].fillna(0).clip(lower=1)), bins=40, color="#76B7B2")
ax[1].set_title("Library size per sample"); ax[1].set_xlabel("log10(total counts)")
plt.tight_layout(); plt.show()

---
# Part 5 · Whole blood only

🎯 **Goal:** keep only genuine **whole-blood** samples. No PBMC, no sorted cells.

💡 **Why two layers of defence:** a PBMC or sorted-cell sample has a very different transcriptome, so mixing it into a whole-blood cohort would inject a fake effect.
1. **Label check:** `sample_source` must be *exactly* one of `BLOOD_VALUES` (so `B cells`, `Neutrophils` and `skin` in NCI-FNL are excluded).
2. **Sneaky check:** even if labelled "blood", the sample name / treatment / cohort text is searched for cell-fraction words. Anything caught is excluded and printed.

The label `blood` cannot by itself tell whole blood from PBMC; the sneaky check can only catch what the names reveal.

In [ ]:
# ---- 2. whole blood only (no PBMC, no sorted cell fractions) ----------------
src = rmeta["sample_source"].astype(str).str.strip().str.lower()
print("\nsample source values (per sheet):")
print(pd.crosstab(rmeta["source_sheet"].str[:30], src).to_string())
NOT_WHOLE_BLOOD = re.compile(r"pbmc|mononuclear|monocyte|neutrophil|lymphocyte|"
                             r"t[\s_-]?cell|b[\s_-]?cell|nk[\s_-]?cell|cd\d|sorted|isolat", re.I)
blob = (rmeta["sample_id"].astype(str) + " " + src + " " + rmeta["treatment"].astype(str)
        + " " + rmeta["sequencing_cohort"].astype(str))
sneaky = src.isin(BLOOD_VALUES) & blob.str.contains(NOT_WHOLE_BLOOD)
print(f"\nlabelled blood but name suggests a cell fraction: {int(sneaky.sum())}")
if sneaky.any():
    print(rmeta.loc[sneaky, ["sample_id", "sample_source"]].head(20).to_string())
rmeta["is_blood"] = src.isin(BLOOD_VALUES) & ~sneaky

---
# Part 6 · Dates (for age and ordering)

🎯 **Goal:** a best-available draw date per sample, used only for computing age and for choosing the *earliest* sample.

💡 **Why a priority order:** a real `rna_collection_date` is preferred. If missing, a date is mined from the sample name (e.g. `121003_121016`). That stamp is a **sequencing batch date**, shared by PRE and POST samples, so it is only a rough proxy. Its provenance is stored in `age_date_source`. Dates outside 2008-today are rejected as typos.

In [ ]:
# ---- 3. date (for age and ordering only) -----------------------------------
def date_from_name(s):
    m = re.search(r"(?<!\d)(\d{6})_\d{6}(?!\d)", s) or re.search(r"(?<!\d)(\d{6})(?!\d)", s)
    if not m: return pd.NaT
    try: d = pd.to_datetime(m.group(1), format="%y%m%d")
    except ValueError: return pd.NaT
    return d if DATE_MIN <= d <= DATE_MAX else pd.NaT

coll = pd.to_datetime(rmeta["rna_collection_date"], errors="coerce")
coll = coll.where((coll >= DATE_MIN) & (coll <= DATE_MAX))
from_name = rmeta["sample_id"].map(date_from_name)
rmeta["_date"] = coll.fillna(from_name)
rmeta["age_date_source"] = np.where(coll.notna(), "rna_collection_date",
                            np.where(from_name.notna(), "sample_name (approx)", "none"))

---
# Part 7 · Join patient metadata

🎯 **Goal:** decide *which patient* each sample belongs to, then attach demographics and age.

💡 **Why a careful ladder** (in order of trust): exact **Enrollment ID**; the **HC fix** (`1126` is `R1126` in the metadata); **initials, only when safe** (RNA ID blank or itself initials-derived, and initials unique); a **diagnosis sanity check** on initials matches (`initials_conflict` stays unmatched; compatible wording is allowed as `initials_dx_compat`). A real-looking ID the metadata lacks returns `id_not_in_metadata` and is **never guessed**. `demographics_trusted` is true only for the two strong ID methods.

In [ ]:
# ---- 4. join patient metadata ----------------------------------------------
HEALTHY = re.compile(r"^hc$|health|control", re.I)
def match_meta(r):
    k = normalize_id(r["enrollment_id_rna"])
    ini = str(r["initials_rna"]).strip().upper()
    ini_n = normalize_id(ini)
    if k and k in meta_lookup: return meta_lookup[k], "enrollment_id"
    if k.isdigit() and ("R" + k) in meta_lookup:           # HC '1126' is 'R1126' in the metadata
        return meta_lookup["R" + k], "R_prefix_id"
    # only fall back to initials when the RNA ID is blank or is itself initials-derived (HC-MF, BG-SAF)
    if not (k == "" or (ini_n and k.endswith(ini_n))):
        return -1, "id_not_in_metadata"                    # a real-looking ID that metadata lacks: don't guess
    eid = AUTO_INITIALS_MAP.get(ini)                       # unique initials only
    if eid and eid in meta_lookup:
        i = meta_lookup[eid]
        dx_md, dx_rna = meta_value(i, "dx1"), str(r["diagnosis_rna"])
        same = (norm_colname(dx_md) == norm_colname(dx_rna)
                or (HEALTHY.search(dx_md) and HEALTHY.search(dx_rna)))
        compat = (re.search(r"interferon|aicardi|\bags\b", dx_md, re.I)
                  and re.search(r"\bags\b|aicardi", dx_rna, re.I))
        if dx_md == "" or same: return i, "initials"
        if compat: return i, "initials_dx_compat"
        return -1, "initials_conflict"
    return -1, "none"

mm = rmeta.apply(lambda r: pd.Series(match_meta(r), index=["meta_row", "meta_join"]), axis=1)
rmeta = pd.concat([rmeta, mm], axis=1)
extra_cols = {}
for f, name in [("sex", "sex"), ("race", "race"), ("ethnicity", "ethnicity"),
                ("status", "study_status"), ("dx1", "diagnosis1"), ("dx2", "diagnosis2"),
                ("note", "metadata_note"), ("dob", "_dob"), ("enroll", "_md_enroll")]:
    extra_cols[name] = rmeta["meta_row"].map(lambda i, f=f: meta_value(i, f))
rmeta = pd.concat([rmeta, pd.DataFrame(extra_cols)], axis=1)
rmeta["demographics_trusted"] = rmeta["meta_join"].isin(["enrollment_id", "R_prefix_id"])
print("\nmetadata join:"); print(rmeta["meta_join"].value_counts().to_string())

dob = pd.to_datetime(rmeta["_dob"], errors="coerce")
rmeta["age_yrs"] = ((rmeta["_date"] - dob).dt.days / 365.25).round(1)
rmeta.loc[(rmeta["age_yrs"] < 0) | (rmeta["age_yrs"] > 120), "age_yrs"] = np.nan

---
# Part 8 · Treatment status

🎯 **Goal:** label each sample `pre`, `post`, or `unlabelled`.

💡 **Why this is the most important filter:** drugs (e.g. JAK inhibitors) can suppress an interferon signature almost completely, so a post-treatment sample looks *healthy* and quietly dilutes the disease cohort. The `treatment` text is free-form; a pattern recognises "pre" wording and **any other non-blank text is treated as post-treatment**, the cautious default. (In their data, all POST samples are CANDLE, and their notebook 41 pooled them with untreated samples without a treatment term. Excluding them here avoids that.)

In [ ]:
# ---- 4b. treatment status --------------------------------------------------
PRE_TX = re.compile(r"pre|baseline|bsl|untreat|naive|none|before|screen|off[\s-]?(?:tx|treat)|^v?0$|^day\s*0$", re.I)
tx = rmeta["treatment"].astype(str).str.strip()
is_pre = (tx != "") & tx.str.contains(PRE_TX)
rmeta["is_post_tx"] = (tx != "") & ~is_pre
rmeta["tx_status"] = np.where(rmeta["is_post_tx"], "post", np.where(is_pre, "pre", "unlabelled"))
print("\ntreatment status (per sheet):")
print(pd.crosstab(rmeta["source_sheet"].str[:30], rmeta["tx_status"]).to_string())

### 👀 Checkpoint 8: which treatment strings were called what?

The regex is a heuristic and a wrong call changes who is in your cohort. Scan this table for anything misclassified and fix `PRE_TX` if so.

In [ ]:
display(rmeta.assign(treatment=rmeta["treatment"].replace("", "(blank)"))
        .groupby(["tx_status", "treatment"]).size().rename("n")
        .reset_index().sort_values(["tx_status", "n"], ascending=[True, False]).head(40))

---
# Part 9 · Eligible set, one per patient, and can batch be separated?

🎯 **Goal:** two cohorts from one set of rules, and a table that says which batches can be separated from diagnosis.

💡 **Why two cohorts:**

| Cohort | Used for | Why |
|---|---|---|
| **eligible (all samples)** | differential expression with limma-voom, batch as a term, **patient as a block** (`duplicateCorrelation`) | uses every usable sample; repeated samples from one patient are handled by the block |
| **one per patient** | WGCNA, `mroast`, anything assuming independent samples | repeated measures from one person are not independent |

**Eligibility** (all must hold): whole blood, has counts, **not** post-treatment, a patient identity exists. For diagnoses in `REQUIRE_PRE_FOR_DX` an **unlabelled** sample is excluded rather than assumed pre-treatment. **Choosing one per patient:** explicit `PRE` beats unlabelled; then earliest date (or deepest library if `"deepest"`); library size breaks ties.

**The batch × diagnosis table is the key output of this part.** A batch that holds only one diagnosis cannot be separated from that diagnosis by any model term, and a batch term is only meaningful where the batch contains more than one group. Healthy controls in particular come from a different workbook than the patients, so HC-vs-patient is confounded with source and should be treated as descriptive.

In [ ]:
# ---- 5. eligible set and one sample per patient ----------------------------
rmeta["patient_key"] = [normalize_id(a) or normalize_id(b) or normalize_id(c)
    for a, b, c in zip(rmeta["_md_enroll"], rmeta["enrollment_id_rna"], rmeta["initials_rna"])]
need_pre = rmeta["dx_clean"].astype(str).str.contains(REQUIRE_PRE_FOR_DX, case=False, regex=True)
base = (rmeta["is_blood"] & rmeta["has_counts"] & ~rmeta["is_post_tx"]
        & (rmeta["patient_key"] != ""))
print(f"excluded for lacking a PRE label ({REQUIRE_PRE_FOR_DX}):", int((base & need_pre & ~is_pre).sum()))
elig = base & (~need_pre | is_pre)
rmeta["eligible"] = elig
b = rmeta[elig].copy()
b["_pre_rank"] = (b["tx_status"] != "pre").astype(int)       # explicit PRE beats unlabelled
if KEEP_ONE_PER_PATIENT == "earliest":
    b = b.sort_values(["patient_key", "_pre_rank", "_date", "total_counts"],
                      ascending=[True, True, True, False], na_position="last")
else:
    b = b.sort_values(["patient_key", "_pre_rank", "total_counts"], ascending=[True, True, False])
rmeta["kept"] = rmeta["sample_id"].isin(b.drop_duplicates("patient_key")["sample_id"])
rmeta["one_per_patient"] = rmeta["kept"]
final_meta = rmeta[rmeta["kept"]].copy()           # one per patient
elig_meta = rmeta[rmeta["eligible"]].copy()        # all eligible samples

rmeta["status"] = np.select(
    [~rmeta["has_counts"], ~rmeta["is_blood"], rmeta["is_post_tx"], rmeta["patient_key"] == "",
     need_pre & ~is_pre, ~rmeta["kept"]],
    ["no counts in file", "not whole blood", "post-treatment", "no patient ID",
     "CANDLE/SAVI without PRE label", "eligible, not the chosen sample of this patient"],
    default="kept")
rmeta.loc[rmeta["eligible"] & ~rmeta["kept"], "status"] = "eligible, not the chosen sample of this patient"

print(f"\nsamples: {len(rmeta)} -> {int(rmeta['is_blood'].sum())} blood "
      f"-> {int(elig.sum())} eligible ({elig_meta['patient_key'].nunique()} patients) "
      f"-> {len(final_meta)} one-per-patient")
print("age known (one per patient):", int(final_meta["age_yrs"].notna().sum()), "of", len(final_meta))
for title, col in [("treatment status", "tx_status"), ("metadata join", "meta_join"),
                   ("age date source", "age_date_source")]:
    print(f"\nkept (one per patient) by {title}:")
    print(pd.crosstab(final_meta["dx_clean"].replace("", "(blank)"), final_meta[col]).to_string())

def batch_table(df, title):
    t = pd.crosstab(df["batch"], df["dx_clean"].replace("", "(blank)"))
    n_dx = (t > 0).sum(axis=1)
    t = t.assign(n_diagnoses=n_dx)
    print(f"\n{title}")
    print(t.to_string())
    one = list(t.index[n_dx == 1])
    if one:
        print("batches holding a single diagnosis (cannot be separated from it by a batch term):", one)
    return t

bt_elig = batch_table(elig_meta, "BATCH x DIAGNOSIS, all eligible samples")
bt_kept = batch_table(final_meta, "BATCH x DIAGNOSIS, one per patient")

ini_ct = final_meta.assign(ini=final_meta["initials_rna"].str.upper()).groupby("ini")["patient_key"].agg(list)
print("\nsame initials under more than one patient key (possible duplicate person):")
print(ini_ct[ini_ct.str.len() > 1].to_string() if (ini_ct.str.len() > 1).any() else "  none")

### 👀 Checkpoint 9: the funnel, and batch against diagnosis

**Left:** the fate of every sample. **Right:** all eligible samples by batch, coloured by diagnosis. A batch bar of a single colour is a batch that cannot be separated from that diagnosis; healthy controls are expected to form their own bar.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
rmeta["status"].value_counts().plot.barh(ax=ax[0], color="#4C78A8")
ax[0].invert_yaxis(); ax[0].set_title(f"Fate of all {len(rmeta)} samples")
bt_elig.drop(columns="n_diagnoses").plot.barh(stacked=True, ax=ax[1])
ax[1].invert_yaxis(); ax[1].set_title("Eligible samples by batch and diagnosis"); ax[1].set_ylabel("")
ax[1].legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()

---
# Part 10 · Gene lengths → RPKM and TPM (display values)

🎯 **Goal:** length-normalised expression for **plots and for comparison with the JDM lab's RPKM files**.

💡 **Why these are display values, not the analysis input:** counts-based tools (limma-voom, DESeq2, edgeR) model the raw counts directly and handle depth themselves, with batch as a term. RPKM/TPM remain useful for heatmaps, for the check against the JDM lab's numbers (notebook 3), and where only RPKM exists.

| Measure | Formula | Properties |
|---|---|---|
| **RPKM** | counts ÷ gene length (kb) ÷ library size (millions) | depth then length; column sums differ between samples |
| **TPM** | (counts ÷ length) ÷ Σ(counts ÷ length) × 10⁶ | every sample sums to 1,000,000; better across samples |

**Gene length** is the non-overlapping exon union per gene from a GENCODE GTF. The notebook first looks for a cached `gene_lengths.csv` (new folder, then the old run's folder), then any `.gtf` on the mounts, then downloads GENCODE v19. Version suffixes are stripped if IDs don't match, and the **match rate is printed**; if it is low, the GTF release doesn't match your counts. Raw counts are written *before* this step can fail.

In [ ]:
# ---- 6. gene lengths -> RPKM and TPM ---------------------------------------
def gtf_gene_lengths(path):
    op = gzip.open if path.endswith(".gz") else open
    rows = []
    with op(path, "rt") as fh:
        for line in fh:
            if line[0] == "#": continue
            f = line.split("\t")
            if len(f) > 8 and f[2] == "exon":
                rows.append((re.search(r'gene_id "([^"]+)"', f[8]).group(1), int(f[3]), int(f[4])))
    ex = pd.DataFrame(rows, columns=["g", "s", "e"]).sort_values(["g", "s"])
    prev = ex.groupby("g")["e"].cummax().groupby(ex["g"]).shift().fillna(0)
    return ((ex["e"] - np.maximum(ex["s"] - 1, prev)).clip(lower=0)).groupby(ex["g"]).sum()

len_csv = os.path.join(OUT_DIR, "gene_lengths.csv")
if not os.path.exists(len_csv) and os.path.exists(LEGACY_LENGTHS):
    shutil.copy2(LEGACY_LENGTHS, len_csv)
    print("reused gene lengths from the previous run:", LEGACY_LENGTHS)
lengths = pd.read_csv(len_csv, index_col=0).iloc[:, 0] if os.path.exists(len_csv) else None
if lengths is None:
    gtf = GTF_PATH or next(iter(sum((glob.glob(f"{r}/**/*.gtf*", recursive=True)
            for r in ["/mnt/file-systems", "/mnt/session_data", "/home/jovyan"]), [])), "")
    if not gtf:
        try:
            gtf = "/home/jovyan/gencode.v19.annotation.gtf.gz"
            print("downloading", GTF_URL); urllib.request.urlretrieve(GTF_URL, gtf)
        except Exception as e:
            print("could not download GTF:", e); gtf = ""
    if gtf:
        lengths = gtf_gene_lengths(gtf); lengths.to_csv(len_csv, header=["length_bp"])

In [ ]:
# ---- raw counts first (these never depend on gene lengths) -------------------
def with_names(mat):
    o = mat.copy(); o.insert(0, "Gene_Name", o.index.map(gene_names)); return o

cc = counts[list(final_meta["sample_id"])]                       # one per patient
ce = counts[list(elig_meta["sample_id"])]                        # all eligible
with_names(cc).to_csv(os.path.join(OUT_DIR, "rnaseq_counts_raw.csv"))
with_names(ce).to_csv(os.path.join(OUT_DIR, "rnaseq_counts_raw_eligible.csv"))
with_names(counts[list(rmeta.loc[rmeta["has_counts"], "sample_id"])]).to_csv(
    os.path.join(OUT_DIR, "rnaseq_counts_all_samples.csv.gz"), compression="gzip")

if lengths is None:
    print("\nNo gene lengths available: raw counts written, RPKM skipped. "
          "Put a .gtf(.gz) on the mount and set GTF_PATH.")
else:
    versioned = cc.index.isin(lengths.index).mean() > 0.5
    lkp, key = lengths.copy(), cc.index
    if not versioned:
        lkp.index = lkp.index.str.split(".").str[0]; lkp = lkp[~lkp.index.duplicated()]
        key = cc.index.str.split(".").str[0]
    L = pd.Series(lkp.reindex(key).values / 1000.0, index=cc.index)
    print(f"\ngene length found for {L.notna().mean():.1%} of {len(L)} genes")
    lib = cc.sum()                                   # full library size, before dropping genes
    ok = L.notna() & (L > 0)
    c2 = cc[ok]
    rpkm = c2.div(L[ok], axis=0).div(lib / 1e6, axis=1)
    rate = c2.div(L[ok], axis=0); tpm = rate.div(rate.sum(), axis=1) * 1e6
    for nm, mat in (("rpkm", rpkm), ("tpm", tpm)):
        with_names(mat).to_csv(os.path.join(OUT_DIR, f"rnaseq_{nm}.csv"))

### 👀 Checkpoint 10: sanity-check the normalisation

**Left:** TPM column sums should all be exactly 1,000,000. **Right:** per-sample log TPM distributions should overlay closely; one curve far off is an outlier worth investigating.

In [ ]:
if lengths is not None:
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
    ax[0].hist(tpm.sum(), bins=20, color="#4C78A8"); ax[0].set_title("TPM column sums (should be 1e6)")
    for s in tpm.columns[:60]:
        ax[1].hist(np.log2(tpm[s] + 1), bins=60, histtype="step", alpha=.4, density=True)
    ax[1].set_title("log2(TPM+1) per sample (first 60)")
    plt.tight_layout(); plt.show()
else:
    print("no gene lengths, nothing to plot")

---
# Part 11 · Write outputs and final audit

🎯 **Goal:** write the sample sheets and supporting CSVs, then audit the weakest matches.

| File | Use |
|---|---|
| `rnaseq_samples.csv` + `rnaseq_counts_raw.csv` | **one per patient**: WGCNA, `mroast`, independence-assuming methods |
| `rnaseq_samples_eligible.csv` + `rnaseq_counts_raw_eligible.csv` | **all eligible samples**: limma-voom with `batch` as a term and `patient_key` as a block |
| `rnaseq_counts_all_samples.csv.gz` | every sample with counts, for cross-checks against the JDM lab (notebook 3) |
| `rnaseq_rpkm.csv` / `rnaseq_tpm.csv` | display values for the one-per-patient cohort |
| `rnaseq_samples_ALL_with_flags.csv` | **every** sample with every flag and a `status`: the audit trail |
| `unmatched_or_weak_metadata.csv` | kept samples whose demographics are not trusted |
| `batch_by_diagnosis_eligible.csv` | the batch × diagnosis table |
| `gene_lengths.csv` | cached gene lengths |

In [ ]:
# ---- 7. sample sheets (no birth date, no raw dates) ------------------------
cols = ["sample_id", "patient_key", "enrollment_id_rna", "initials_rna", "meta_join",
        "demographics_trusted", "sample_source", "treatment", "tx_status", "diagnosis_rna",
        "dx_clean", "diagnosis1", "diagnosis2", "study_status", "sex", "race", "ethnicity",
        "age_yrs", "age_date_source", "sequencing_facility", "sequencing_cohort",
        "facility", "batch", "total_counts", "source_file", "source_sheet"]
final_meta[cols].to_csv(os.path.join(OUT_DIR, "rnaseq_samples.csv"), index=False)
elig_meta.assign(one_per_patient=elig_meta["kept"])[cols + ["one_per_patient"]].to_csv(
    os.path.join(OUT_DIR, "rnaseq_samples_eligible.csv"), index=False)
final_meta.loc[~final_meta["demographics_trusted"],
               ["sample_id", "enrollment_id_rna", "initials_rna", "meta_join", "dx_clean", "source_file"]
               ].to_csv(os.path.join(OUT_DIR, "unmatched_or_weak_metadata.csv"), index=False)
rmeta.drop(columns=["_dob", "_date", "_md_enroll", "meta_row"]).to_csv(
    os.path.join(OUT_DIR, "rnaseq_samples_ALL_with_flags.csv"), index=False)
bt_elig.to_csv(os.path.join(OUT_DIR, "batch_by_diagnosis_eligible.csv"))

# short generated log, so it cannot go stale
def finfo(p):
    return (f"{os.path.basename(p)}  ({os.path.getsize(p)/1e6:.1f} MB, modified "
            f"{datetime.datetime.fromtimestamp(os.path.getmtime(p)):%Y-%m-%d})")
log = ["# RNA-seq processing log (notebook 2, v2)",
       f"Generated {datetime.datetime.now():%Y-%m-%d %H:%M}.\n", "## Inputs"]
log += [f"- {finfo(p)}" for p in RNA_FILES + [METADATA_PATH]]
log += ["", "## Units read (every sheet)", "```", units_df.to_string(index=False), "```", "",
        "## Rules, in order",
        "0. samples with empty counts excluded; 1. whole blood only (exact label + name check);",
        "2. post-treatment excluded; 3. CANDLE/SAVI need an explicit PRE label;",
        f"4. one per patient: explicit PRE, then '{KEEP_ONE_PER_PATIENT}', then deepest library.",
        "Batch = Sequencing_Facility (NIAMS split by Sequencing_Cohort).", "",
        f"Samples read: {len(rmeta)}; eligible: {int(elig.sum())}; one per patient: {len(final_meta)}.", "",
        "## Fate of every sample", "```", rmeta["status"].value_counts().to_string(), "```", "",
        "## Batch x diagnosis (eligible)", "```", bt_elig.to_string(), "```"]
open(os.path.join(OUT_DIR, "PROCESSING_LOG_notebook2.md"), "w").write("\n".join(log))
print("\nwritten to", OUT_DIR, ":")
for f in sorted(os.listdir(OUT_DIR)):
    fp = os.path.join(OUT_DIR, f)
    if os.path.isfile(fp): print(f"  {f:46s} {os.path.getsize(fp)/1e6:8.1f} MB")

### Final audit: weak or unmatched kept samples

In [ ]:
k = final_meta
w = k[~k["meta_join"].isin(["enrollment_id", "R_prefix_id"])]
print(pd.crosstab(k["dx_clean"], k["meta_join"]).to_string())
print(f"\nweak / unmatched: {len(w)}")
print(w[["sample_id", "enrollment_id_rna", "initials_rna", "meta_join", "patient_key", "dx_clean", "sex", "age_yrs"]]
      .assign(sample_id=w["sample_id"].str[:30]).to_string(index=False))

---
## ✅ Done. What to check, and where to go next

1. **Part 4:** do the per-sheet sample counts look right (compare with BCM-HGSC 11, NCI-FNL 20, NIAID-NCI 15, NIAMS 126)?
2. **Checkpoint 8:** confirm the pre/post calls. This single regex decides cohort membership.
3. **Part 9:** read the batch × diagnosis table. Which batches hold one diagnosis only?
4. Work through the weak / unmatched list; fix IDs at the source and re-run.
5. Confirm the gene-length match rate in Part 10 is high (about 95% or more).
6. **Next:** notebook 3 treats the JDM lab as its own study; notebook 4 builds WGCNA per study. Counts-based tests should use `rnaseq_counts_raw_eligible.csv` with `batch` in the design and `patient_key` as the block.